# OptiCrop: Machine Learning Model Selection & Comparison

This notebook demonstrates the preprocessing, training, evaluation, and comparison of multiple machine learning models for the **OptiCrop** intelligent agricultural recommendation system. We compare the following algorithms:

1. **Logistic Regression**
2. **K-Nearest Neighbors (KNN)**
3. **Decision Tree Classifier**
4. **Random Forest Classifier**
5. **K-Means Clustering** (evaluated by mapping clusters to dominant target labels)

We evaluate their performance based on soil and climate parameters: Nitrogen (N), Phosphorous (P), Potassium (K), Temperature, Humidity, pH Level, and Rainfall to select and persist the best-performing model for deployment.

### 1. Imports and Data Preprocessing

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans

# Read the dataset
df = pd.read_csv('dataset.csv')
df = df.dropna()
print(f"Dataset shape: {df.shape}")
df.head()

### 2. Feature Scaling and Train-Test Split

In [ ]:
FEATURE_COLUMNS = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
X = df[FEATURE_COLUMNS]
y = df["crop"]

# Scale Features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split into Train and Test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

### 3. Model Training & Evaluation

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

accuracies = {}
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    accuracies[name] = acc
    trained_models[name] = model
    print(f"{name} Accuracy: {acc * 100:.2f}%")

### 4. K-Means Clustering Integration

Here, we perform K-Means Clustering on the dataset with `k=42` (matching the number of unique crop categories). We evaluate K-Means as a recommendation system by mapping each cluster to the most frequent true crop in that cluster.

In [ ]:
n_classes = len(y.unique())
kmeans = KMeans(n_clusters=n_classes, random_state=42, n_init=10)
kmeans.fit(X_train)

# Map clusters to true classes
cluster_labels = kmeans.labels_
cluster_to_crop = {}
for cluster in range(n_classes):
    indices = np.where(cluster_labels == cluster)[0]
    if len(indices) > 0:
        crops_in_cluster = y_train.iloc[indices]
        most_common = crops_in_cluster.mode()[0]
        cluster_to_crop[cluster] = most_common
    else:
        cluster_to_crop[cluster] = "unknown"

# Predict on test set
test_clusters = kmeans.predict(X_test)
kmeans_preds = [cluster_to_crop.get(c, "unknown") for c in test_clusters]
kmeans_acc = accuracy_score(y_test, kmeans_preds)
accuracies["K-Means Clustering"] = kmeans_acc
print(f"K-Means Cluster Mapping Accuracy: {kmeans_acc * 100:.2f}%")

### 5. Visualizing Model Performance

In [ ]:
# Accuracy Comparison Chart
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")
bar_data = pd.DataFrame({
    "Algorithm": list(accuracies.keys()),
    "Accuracy": [acc * 100 for acc in accuracies.values()]
})
ax = sns.barplot(x="Algorithm", y="Accuracy", data=bar_data, palette="viridis")
plt.title("Model Accuracy Comparison", fontsize=16, fontweight='bold', pad=15)
plt.ylabel("Accuracy (%)", fontsize=12)
plt.ylim(0, 105)
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}%", 
                (p.get_x() + p.get_width() / 2., p.get_height() + 1.5), 
                ha='center', va='center', fontsize=10, color='black', 
                fontweight='bold', xytext=(0, 5), textcoords='offset points')
plt.show()

In [ ]:
# Confusion Matrix for the Best Model (Random Forest)
best_model_name = "Random Forest"
best_model = trained_models[best_model_name]
y_pred_best = best_model.predict(X_test)
labels = sorted(list(y_test.unique()))
cm = confusion_matrix(y_test, y_pred_best, labels=labels)

plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=labels, yticklabels=labels)
plt.title(f"Confusion Matrix - {best_model_name}", fontsize=18, fontweight='bold', pad=20)
plt.ylabel("True Crop Label", fontsize=14)
plt.xlabel("Predicted Crop Label", fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance (Random Forest)
plt.figure(figsize=(10, 6))
importances = best_model.feature_importances_
indices = np.argsort(importances)[::-1]
feature_ranking = [FEATURE_COLUMNS[i] for i in indices]

sns.barplot(x=importances[indices] * 100, y=feature_ranking, palette="mako")
plt.title("Random Forest - Feature Importance", fontsize=16, fontweight='bold', pad=15)
plt.xlabel("Importance Score (%)", fontsize=12)
plt.ylabel("Soil/Environmental Parameter", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# PCA 2D Scatter Plot of K-Means Clusters
plt.figure(figsize=(10, 8))
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans.predict(X_scaled), cmap="tab20", s=25, alpha=0.8)
plt.colorbar(scatter, label="Cluster ID")
plt.title("K-Means Clustering Visualization (2D PCA)", fontsize=16, fontweight='bold', pad=15)
plt.xlabel("Principal Component 1", fontsize=12)
plt.ylabel("Principal Component 2", fontsize=12)
plt.tight_layout()
plt.show()

### 6. Model Persistence

In [ ]:
# Save Model and Scaler
with open('model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Best model (Random Forest) and scaler saved successfully!")